# Heart Failure Detection System Using Machine Learning
### College Capstone Engineering Project
**Team Members:**
- **Adhithyan JS** (Roll No. 7)
- **Evin Saj Abraham** (Roll No. 29)
- **Farhana H** (Roll No. 30)
- **Ridhin Krishna M** (Roll No. 53)

---

## 1. Project Abstract & Objectives
Heart failure and cardiovascular diseases (CVDs) are the leading causes of global mortality, responsible for an estimated 17.9 million deaths annually. Early risk identification through non-invasive clinical indicators provides vital decision-support for physicians.

### Project Goals:
1. Conduct exploratory data analysis (EDA) to determine key physiological and electrocardiographic predictors of heart disease.
2. Address real-world clinical data quality anomalies (e.g., zero-valued serum cholesterol and resting blood pressure) without data leakage.
3. Train, optimize, and cross-validate four candidate classification architectures:
   - Logistic Regression
   - Decision Tree
   - Random Forest
   - Support Vector Machine (SVM)
4. Prioritize **Recall (Sensitivity)** and **F1-Score** to penalize dangerous false negatives (missed cardiac disease).
5. Build an end-to-end reproducible scikit-learn `Pipeline` and interactive clinical decision-support interface.

## 2. Environment Setup & Library Imports

In [ ]:
import os
import sys
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Scikit-Learn tools
from sklearn.model_selection import train_test_split, StratifiedKFold, GridSearchCV
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.feature_selection import mutual_info_classif
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, roc_curve, confusion_matrix, classification_report
)
import joblib

# Styling setup
plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
%matplotlib inline
print("Libraries loaded successfully.")

## 3. Dataset Loading & Schema Inspection
The dataset comprises 918 records synthesized from 5 renowned clinical cardiac datasets (Cleveland, Hungarian, Switzerland, Long Beach VA, Statlog).
We verify schema, column data types, missing entries, and duplicates.

In [ ]:
# Load dataset
data_path = Path('../data/heart.csv')
if not data_path.exists():
    data_path = Path('data/heart.csv')

df = pd.read_csv(data_path)
print(f"Dataset Shape: {df.shape[0]} rows, {df.shape[1]} columns")
print("\nColumn Data Types:")
print(df.dtypes)
print(f"\nDuplicate rows: {df.duplicated().sum()}")
df.head(10)

## 4. Clinical Data Quality Audit & Remediation Strategy
### Identified Anomalies:
1. **Serum Cholesterol = 0 mg/dL:**
   - There are **172 zero entries** in `Cholesterol` (~18.7% of the cohort).
   - In living humans, serum cholesterol cannot physiologically be zero. These represent unmeasured/missing laboratory tests.
   - **Remediation:** Convert `0` to `NaN`, impute using the **median of the training set**, and introduce a binary flag `Cholesterol_missing = 1` to capture potential admission bias without leakage.
2. **Resting Blood Pressure = 0 mm Hg:**
   - Exactly **1 record** contains `RestingBP = 0`. Handled via median imputation.
3. **Class Balance:**
   - 508 Heart Disease (55.3%) vs. 410 Normal (44.7%). Classes are naturally balanced; no SMOTE or resampling needed.

In [ ]:
chol_zeros = (df['Cholesterol'] == 0).sum()
bp_zeros = (df['RestingBP'] == 0).sum()
print(f"Total zero values in Cholesterol: {chol_zeros} ({chol_zeros/len(df)*100:.2f}%)")
print(f"Total zero values in RestingBP: {bp_zeros}")

# Inspect class balance
target_counts = df['HeartDisease'].value_counts()
print("\nTarget Class Balance:")
print(target_counts)
print(f"Positive Ratio: {target_counts[1]/len(df):.2%}")

## 5. Exploratory Data Analysis (EDA)
Visualizing distributions, relationships, and clinical patterns.

In [ ]:
# Target Distribution Plot
fig, ax = plt.subplots(figsize=(6, 4))
df['HeartDisease'].value_counts().plot(kind='bar', color=['#e74c3c', '#3498db'], ax=ax, rot=0)
ax.set_xticklabels(['Heart Disease (1)', 'Normal (0)'])
ax.set_title("Distribution of Target: Heart Disease vs Normal", fontsize=12, fontweight='bold')
ax.set_ylabel("Patient Count")
plt.tight_layout()
plt.show()

In [ ]:
# Categorical Features Stratified by Diagnosis
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
cat_cols = ['ChestPainType', 'ST_Slope', 'ExerciseAngina', 'Sex']
for idx, col in enumerate(cat_cols):
    ax = axes[idx // 2, idx % 2]
    sns.countplot(data=df, x=col, hue='HeartDisease', palette=['#3498db', '#e74c3c'], ax=ax)
    ax.set_title(f"{col} vs Heart Disease", fontweight='bold')
    ax.legend(['Normal (0)', 'Heart Disease (1)'])
plt.tight_layout()
plt.show()

In [ ]:
# Continuous Features Boxplots
numeric_cols = ['Age', 'RestingBP', 'Cholesterol', 'MaxHR', 'Oldpeak']
fig, axes = plt.subplots(1, 5, figsize=(18, 4))
for idx, col in enumerate(numeric_cols):
    sns.boxplot(data=df, x='HeartDisease', y=col, palette=['#3498db', '#e74c3c'], ax=axes[idx])
    axes[idx].set_title(f"{col} by Diagnosis")
    axes[idx].set_xticklabels(['Normal', 'Heart Disease'])
plt.tight_layout()
plt.show()

In [ ]:
# Numerical Correlation Heatmap
plt.figure(figsize=(8, 6))
sns.heatmap(df[numeric_cols + ['HeartDisease']].corr(), annot=True, cmap='coolwarm', vmin=-1, vmax=1, fmt='.2f')
plt.title("Pearson Correlation Heatmap", fontsize=13, fontweight='bold')
plt.show()

## 6. Feature Selection & Importance Evaluation
We analyze feature contributions using:
1. Pearson Correlation with Target
2. Mutual Information Scores
3. Random Forest Gini Importance

**Conclusion:** All 11 features contain substantial clinical value and predictive signal. Retaining all 11 features prevents information loss while keeping the model compact and interpretable.

In [ ]:
# Prepare encoded X for mutual info
X_raw = df.drop(columns=['HeartDisease'])
y_raw = df['HeartDisease']

# One-hot encode string features for mutual info evaluation
X_encoded = pd.get_dummies(X_raw, drop_first=True)
mi_scores = mutual_info_classif(X_encoded, y_raw, random_state=42)
mi_series = pd.Series(mi_scores, index=X_encoded.columns).sort_values(ascending=False)

plt.figure(figsize=(10, 6))
mi_series.plot(kind='barh', color='#2b5c8f')
plt.title("Mutual Information Ranking with Heart Disease", fontsize=12, fontweight='bold')
plt.xlabel("Mutual Information Score")
plt.gca().invert_yaxis()
plt.tight_layout()
plt.show()

## 7. Leakage-Free Preprocessing Pipeline
We encapsulate custom data cleaning, median imputation, standard scaling, and one-hot encoding into a unified scikit-learn `Pipeline`.

In [ ]:
from sklearn.base import BaseEstimator, TransformerMixin

class ClinicalDataCleaner(BaseEstimator, TransformerMixin):
    def __init__(self, cols_with_zeros=['Cholesterol', 'RestingBP']):
        self.cols_with_zeros = cols_with_zeros
        
    def fit(self, X, y=None):
        return self
        
    def transform(self, X):
        X_out = X.copy()
        if not isinstance(X_out, pd.DataFrame):
            X_out = pd.DataFrame(X_out)
        if 'Cholesterol' in X_out.columns:
            chol = pd.to_numeric(X_out['Cholesterol'], errors='coerce')
            X_out['Cholesterol_missing'] = ((chol == 0) | (chol.isna())).astype(int)
        for col in self.cols_with_zeros:
            if col in X_out.columns:
                series = pd.to_numeric(X_out[col], errors='coerce')
                X_out[col] = series.replace(0, np.nan)
        return X_out

# Split dataset
X = df.drop(columns=['HeartDisease'])
y = df['HeartDisease']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.20, stratify=y, random_state=42)

print(f"Train Set: {X_train.shape[0]} | Test Set: {X_test.shape[0]}")

## 8. Model Training & Stratified 5-Fold Cross-Validation Tuning
We benchmark four models:
1. **Logistic Regression** (Linear baseline)
2. **Decision Tree** (Non-linear interpretable tree)
3. **Random Forest** (Ensemble bagging)
4. **Support Vector Machine** (Kernelized margin separator)

In [ ]:
numeric_features = ['Age', 'RestingBP', 'Cholesterol', 'MaxHR', 'Oldpeak']
categorical_features = ['Sex', 'ChestPainType', 'RestingECG', 'ExerciseAngina', 'ST_Slope']
binary_features = ['FastingBS', 'Cholesterol_missing']

preprocessor = ColumnTransformer(
    transformers=[
        ('num', Pipeline([('imputer', SimpleImputer(strategy='median')), ('scaler', StandardScaler())]), numeric_features),
        ('cat', Pipeline([('encoder', OneHotEncoder(handle_unknown='ignore', sparse_output=False))]), categorical_features),
        ('bin', Pipeline([('imputer', SimpleImputer(strategy='most_frequent'))]), binary_features)
    ]
)

models = {
    'Logistic Regression': (LogisticRegression(random_state=42), {
        'classifier__C': [0.01, 0.1, 1.0, 10.0],
        'classifier__solver': ['lbfgs', 'liblinear']
    }),
    'Decision Tree': (DecisionTreeClassifier(random_state=42), {
        'classifier__max_depth': [3, 5, 7, None],
        'classifier__min_samples_split': [2, 5, 10]
    }),
    'Random Forest': (RandomForestClassifier(random_state=42), {
        'classifier__n_estimators': [50, 100, 200],
        'classifier__max_depth': [4, 6, 8, None],
        'classifier__min_samples_split': [2, 5]
    }),
    'SVM': (SVC(probability=True, random_state=42), {
        'classifier__C': [0.1, 1.0, 10.0],
        'classifier__kernel': ['linear', 'rbf']
    })
}

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
best_estimators = {}
cv_results = {}

for name, (clf, grid) in models.items():
    pipe = Pipeline([
        ('cleaner', ClinicalDataCleaner()),
        ('preprocessor', preprocessor),
        ('classifier', clf)
    ])
    gs = GridSearchCV(pipe, grid, scoring='f1', cv=cv, n_jobs=-1)
    gs.fit(X_train, y_train)
    best_estimators[name] = gs.best_estimator_
    cv_results[name] = gs.best_score_
    print(f"{name:20s} | Best CV F1-Score: {gs.best_score_:.4f}")

## 9. Independent Test Set Evaluation & Model Comparison
We evaluate all four models on the held-out 20% test partition (N=184).

In [ ]:
results = []
for name, model in best_estimators.items():
    y_pred = model.predict(X_test)
    y_prob = model.predict_proba(X_test)[:, 1]
    
    acc = accuracy_score(y_test, y_pred)
    prec = precision_score(y_test, y_pred)
    rec = recall_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred)
    auc = roc_auc_score(y_test, y_prob)
    cm = confusion_matrix(y_test, y_pred)
    tn, fp, fn, tp = cm.ravel()
    
    results.append({
        'Model': name,
        'Accuracy': acc,
        'Precision': prec,
        'Recall': rec,
        'F1-Score': f1,
        'ROC-AUC': auc,
        'False Negatives': fn,
        'CV F1 (Mean)': cv_results[name]
    })

results_df = pd.DataFrame(results).sort_values(by=['Recall', 'F1-Score'], ascending=[False, False])
display(results_df)

In [ ]:
# Comparative ROC Curves
plt.figure(figsize=(8, 6))
colors = ['#1f77b4', '#ff7f0e', '#2ca02c', '#d62728']
for (name, model), color in zip(best_estimators.items(), colors):
    y_prob = model.predict_proba(X_test)[:, 1]
    fpr, tpr, _ = roc_curve(y_test, y_prob)
    auc = roc_auc_score(y_test, y_prob)
    plt.plot(fpr, tpr, label=f"{name} (AUC = {auc:.3f})", color=color, lw=2)

plt.plot([0, 1], [0, 1], 'k--', lw=1.5, label="Chance")
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate (Recall)")
plt.title("ROC Curves Comparison on Held-Out Test Set", fontsize=13, fontweight='bold')
plt.legend(loc='lower right')
plt.tight_layout()
plt.show()

In [ ]:
# Confusion Matrix of Top Model
best_name = results_df.iloc[0]['Model']
best_pipe = best_estimators[best_name]
y_pred_best = best_pipe.predict(X_test)
cm_best = confusion_matrix(y_test, y_pred_best)

plt.figure(figsize=(6, 5))
sns.heatmap(cm_best, annot=True, fmt='d', cmap='Blues',
            xticklabels=['Normal', 'Heart Disease'],
            yticklabels=['Normal', 'Heart Disease'])
plt.title(f"Confusion Matrix: {best_name}\n(Minimized False Negatives)", fontweight='bold')
plt.xlabel("Predicted Diagnosis")
plt.ylabel("Actual Diagnosis")
plt.tight_layout()
plt.show()

## 10. Clinical Model Selection & Pipeline Serialization
- In cardiovascular diagnosis, **False Negatives** (failing to identify a patient with heart disease) pose critical clinical risk.
- Therefore, the model with the highest **Recall** and **F1-Score** is selected for clinical deployment.
- The entire leak-free pipeline is serialized via `joblib` for use in the Streamlit web application.

In [ ]:
# Save best model pipeline
output_model_path = Path('../models/best_pipeline.joblib')
if not output_model_path.parent.exists():
    output_model_path = Path('models/best_pipeline.joblib')
output_model_path.parent.mkdir(parents=True, exist_ok=True)

joblib.dump(best_pipe, output_model_path)
print(f"Best pipeline ({best_name}) successfully exported to: {output_model_path}")